In [1]:
import requests
import json
from time import sleep
import pandas as pd
import os

In [2]:
df = pd.read_csv('wq_dataset.csv')
df = df[df.type=='MATRIX'].reset_index(drop=True)
df.head()

,id,description,dataset,category,subcategory,region,delay,universe,type,coverage,userCount,alphaCount,themes
0,adv20,Average daily volume in past 20 days,"{'id': 'pv1', 'name': 'Price Volume Data for E...","{'id': 'pv', 'name': 'Price Volume'}","{'id': 'pv-price-volume', 'name': 'Price Volume'}",USA,1,TOP3000,MATRIX,1.0000,8243,63557,[]
1,anl4_adjusted_netincome_ft,Adjusted net income - forecast type (revision/...,"{'id': 'analyst4', 'name': 'Analyst Estimate D...","{'id': 'analyst', 'name': 'Analyst'}","{'id': 'analyst-analyst-estimates', 'name': 'A...",USA,1,TOP3000,MATRIX,0.8706,5434,10937,[]
2,anl4_bvps_flag,Book value per share - forecast type (revision...,"{'id': 'analyst4', 'name': 'Analyst Estimate D...","{'id': 'analyst', 'name': 'Analyst'}","{'id': 'analyst-analyst-estimates', 'name': 'A...",USA,1,TOP3000,MATRIX,0.8223,1369,2231,[]
3,anl4_bvps_high,"Book value - the highest estimation, per share","{'id': 'analyst4', 'name': 'Analyst Estimate D...","{'id': 'analyst', 'name': 'Analyst'}","{'id': 'analyst-analyst-estimates', 'name': 'A...",USA,1,TOP3000,MATRIX,0.6499,151,644,[]
4,anl4_bvps_low,"Book value - the lowest estimation, per share","{'id': 'analyst4', 'name': 'Analyst Estimate D...","{'id': 'analyst', 'name': 'Analyst'}","{'id': 'analyst-analyst-estimates', 'name': 'A...",USA,1,TOP3000,MATRIX,0.6499,150,552,[]


In [3]:
def calc(regular):
    username, password = 'xxx@gmail.com','xxx'

    s = requests.Session()

    s.auth = (username, password)

    # Send a POST request to the /authentication API
    response = s.post('https://api.worldquantbrain.com/authentication')
    simulation_data = {
        'type': 'REGULAR',
        'settings': {
            'instrumentType': 'EQUITY',
            'region': 'USA',
            'universe': 'TOP1000',
            'delay': 1,
            'decay': 4,
            'neutralization': 'SUBINDUSTRY',
            'truncation': 0.08,
            'pasteurization': 'ON',
            'testPeriod': 'P1Y6M', #P1Y6M
            'unitHandling': 'VERIFY',
            'nanHandling': 'OFF',
            'language': 'FASTEXPR',
            'visualization': False,
        },
        'regular': regular
    }
    simulation_response = s.post('https://api.worldquantbrain.com/simulations', json=simulation_data)


    simulation_progress_url = simulation_response.headers['Location']
    finished = False
    while True:
        simulation_progress = s.get(simulation_progress_url)
        if simulation_progress.headers.get("Retry-After", 0) == 0:
            break
#         print("Sleeping for " + simulation_progress.headers["Retry-After"] + " seconds")
        sleep(float(simulation_progress.headers["Retry-After"]))
#     print("Alpha done simulating, getting alpha details")
    alpha_id = simulation_progress.json()["alpha"]
#     print(alpha_id)
    alpha = s.get("https://api.worldquantbrain.com/alphas/" + alpha_id)
    result = json.loads(alpha.text)['is']
    result['checks'] = regular
    result = pd.DataFrame(pd.Series(result)).T
    return result

In [4]:
regular = f"-returns"
result = calc(regular)
result

,pnl,bookSize,longCount,shortCount,turnover,returns,drawdown,margin,sharpe,fitness,startDate,checks
0,11013935,20000000,485,479,0.8179,0.2226,0.2748,0.000544,1.05,0.55,2019-01-01,-returns


In [5]:
result.columns.tolist()

['pnl',
 'bookSize',
 'longCount',
 'shortCount',
 'turnover',
 'returns',
 'drawdown',
 'margin',
 'sharpe',
 'fitness',
 'startDate',
 'checks']

In [6]:
# ts_delta(x, d)
# ts_zscore(x, d)
# ts_co_kurtosis(y, x, d)
# ts_corr(x, y, d)
# ts_regression(y, x, d, lag = 0, rettype = 0)
# ts_rank(x, d)
# rank(x)
# ts_ir(x, d)



In [7]:
dataname = list(df.id)
print(len(dataname))

1457


In [9]:
filename = 'simstats'
with open(f'{filename}.csv', 'w', encoding='utf-8') as f:
    lines = ','.join(result.columns.tolist())+'\n'
    f.write(lines)
    lines = ','.join(result.values[0].astype(str).tolist())+'\n'
    f.write(lines)

In [10]:
from tqdm import tqdm
resultall = []
for feat in tqdm(dataname[:3]):
    regular = f"ts_delta({feat}, 240)"
    try:
        result = calc(regular)
        print(regular,'success',result.sharpe.values[0],result.fitness.values[0])
        with open(f'{filename}.csv', 'a', encoding='utf-8') as f:
            lines = ','.join(result.values[0].astype(str).tolist())+'\n'
            f.write(lines)

        resultall.append(result)
    except:
        print(regular,'fail')



 33%|███████████████████████████▋                                                       | 1/3 [02:07<04:15, 127.70s/it]

ts_delta(adv20, 240) success -0.16 -0.08


 67%|███████████████████████████████████████████████████████▎                           | 2/3 [05:09<02:39, 159.42s/it]

ts_delta(anl4_adjusted_netincome_ft, 240) success -0.11 -0.04


100%|███████████████████████████████████████████████████████████████████████████████████| 3/3 [08:37<00:00, 172.34s/it]

ts_delta(anl4_bvps_flag, 240) success 0.38 0.26


In [11]:
resultall = pd.concat(resultall).reset_index(drop=True)
# resultall.to_csv('simstats.csv',index=False)

In [12]:
resultall

,pnl,bookSize,longCount,shortCount,turnover,returns,drawdown,margin,sharpe,fitness,startDate,checks
0,-1372385,20000000,438,492,0.081,-0.0277,0.5615,-0.000685,-0.16,-0.08,2019-01-01,"ts_delta(adv20, 240)"
1,-750367,20000000,199,324,0.0452,-0.0152,0.4877,-0.000671,-0.11,-0.04,2019-01-01,"ts_delta(anl4_adjusted_netincome_ft, 240)"
2,2983650,20000000,24,111,0.051,0.0603,0.495,0.002367,0.38,0.26,2019-01-01,"ts_delta(anl4_bvps_flag, 240)"


In [ ]:
import pandas as pd
resultall = pd.read_csv('simstats.csv')

In [15]:
tt = resultall[~(resultall.fitness.isna())]
tt[(tt.fitness.abs()>1)&(tt.sharpe.abs()>1.25)].sort_values(by='fitness',ascending=False)

,pnl,bookSize,longCount,shortCount,turnover,returns,drawdown,margin,sharpe,fitness,startDate,checks
1324,2209268,20000000,1190,1179,0.1643,0.1146,0.0662,0.001395,1.76,1.47,2018-01-20,"rank(ts_delta(pv13_revere_index_value, 63))"
1323,2108098,20000000,1184,1186,0.1685,0.1093,0.0721,0.001298,1.67,1.35,2018-01-20,"rank(ts_delta(pv13_revere_index_cap, 63))"
773,-2954955,20000000,815,1191,0.0697,-0.0597,0.3266,-0.001714,-1.66,-1.15,2018-01-20,"rank(ts_delta(fnd6_newa1v1300_dlc, 63))"
1079,-8515709,20000000,167,805,0.0538,-0.1721,0.8823,-0.006396,-1.49,-1.75,2018-01-20,"rank(ts_delta(fnd6_weburl, 63))"
670,-11646490,20000000,6,5,0.0116,-0.2781,1.2896,-0.048075,-1.34,-2.0,2018-01-20,"rank(ts_delta(fnd6_esopr, 63))"
1331,-17324966,20000000,121,359,0.079,-0.3501,1.7449,-0.00886,-1.51,-2.53,2018-01-20,"rank(ts_delta(pv13_reveremap, 63))"
1204,-928090,20000000,0,0,1.0,-11.6011,0.3541,-0.023202,-1.69,-5.76,2018-01-20,"rank(ts_delta(news_mins_20_chg, 63))"
